# Certified acquisition optimization with a non-Gaussian PDE-conditioned GP

This notebook is a **decision-side PDE experiment** for Decision-Equivalent / Decision-Certified Conditioning.

The goal is **not** to beat PINNs/PIGPs/FlowGP as a PDE solver. Instead, a PDE creates a large non-Gaussian conditioned GP and Bayesian optimization only needs the next action. We ask how much of that conditioning and inference must be resolved before the action is certified.

The experiment uses:

- a reaction-diffusion / screened-Poisson PDE;
- overlapping five-point residual factors;
- a robust non-Gaussian likelihood `log cosh(residual)`;
- ordinary expected improvement (EI);
- a localized BO trust region inside a larger PDE domain;
- a rigorous structural error bound plus a modular Monte-Carlo error allowance.

**Expected default result:** about 50 active PDE factors out of 576 are enough for an EI certificate around 0.06. Sparse-target GP-reference importance sampling retains roughly 85% ESS, while full-target GP-reference importance sampling falls to roughly 12%. A held-out Laplace-preconditioned importance sampler gives roughly 69% ESS and verifies the DEC action.

## 1. Mathematical setup

Let the ordinary data-conditioned GP at the current BO iteration be

\[
P_0(df)=\mathcal N(m,Q^{-1})(df).
\]

The full scientific condition is

\[
E_C(f)=\sum_{j=1}^N e_j(f),
\qquad
\pi_C(df)\propto e^{-E_C(f)}P_0(df).
\]

For expected improvement,

\[
u_x(f)=(f(x)-y_{\rm best})_+,
\qquad
\alpha_C(x)=\mathbb E_{\pi_C}[u_x(f)].
\]

We want an action \(\widehat x\) satisfying

\[
\alpha_C(x_C^\star)-\alpha_C(\widehat x)\le \epsilon
\]

without constructing an accurate approximation to the full posterior.

The PDE is

\[
f-\kappa\Delta f=s.
\]

With unit grid spacing, normalize its five-point residual as

\[
r_{ij}(f)=f_{ij}-c\sum_{(k,\ell)\in\mathcal N(i,j)}f_{k\ell}-b_{ij},
\qquad
c=\frac{\kappa}{1+4\kappa}.
\]

Each PDE location contributes the robust factor

\[
e_{ij}(f)=\gamma\log\cosh\!\left(\frac{r_{ij}(f)}{\tau}\right).
\]

This makes the conditioned posterior genuinely **non-Gaussian**, while keeping the target log-concave and the derivative bounds explicit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.linalg as la
import scipy.stats as st
from pathlib import Path
import gc

OUTPUT_DIR = Path.cwd() / 'dec_pde_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Fast reproducible settings
N_GRID = 24
GAMMA = 0.08
TAU = 0.30
C_PDE = 0.15
Q0, QL = 3.0, 0.6
RIGHT_BIAS = 0.045
PEAK_SEP, PEAK_SIGMA = 5.0, 1.3
LEFT_AMP, RIGHT_AMP = 1.0, 0.96
Y_BEST = 0.55
N_PARTICLES = 6000
FULL_VALIDATION_PARTICLES = 6000
BATCH_ADD = 10
EPS_TARGET = 0.060
DELTA_MC = 0.05
SEED = 702

KAPPA = C_PDE / (1 - 4*C_PDE)
print(f'kappa = {KAPPA:.3f}')

## 2. Build the PDE field, Gaussian reference, and factor graph

The manufactured truth is used only to define the known source term. The Gaussian reference mean has two plausible high-value regions and a small bias toward the right peak. The BO search is restricted to a central trust region; the PDE factors still cover the entire domain.

In [ ]:
def build_problem(n, right_bias=RIGHT_BIAS):
    coords = np.arange(n) - (n - 1) / 2
    X, Y = np.meshgrid(coords, coords, indexing='ij')
    g_left = np.exp(-((X + PEAK_SEP/2)**2 + Y**2)/(2*PEAK_SIGMA**2))
    g_right = np.exp(-((X - PEAK_SEP/2)**2 + Y**2)/(2*PEAK_SIGMA**2))
    truth = LEFT_AMP*g_left + RIGHT_AMP*g_right
    mean = LEFT_AMP*g_left + (RIGHT_AMP + right_bias)*g_right

    d = n*n
    idx = lambda i,j: i*n+j

    # Gaussian Markov random-field precision Q = q0 I + ql * graph Laplacian.
    Q = np.zeros((d,d))
    degree = np.zeros(d)
    for i in range(n):
        for j in range(n):
            u = idx(i,j)
            for di,dj in [(1,0),(0,1)]:
                ii,jj=i+di,j+dj
                if ii<n and jj<n:
                    v=idx(ii,jj); degree[u]+=1; degree[v]+=1
    np.fill_diagonal(Q, Q0 + QL*degree)
    for i in range(n):
        for j in range(n):
            u=idx(i,j)
            for di,dj in [(1,0),(0,1)]:
                ii,jj=i+di,j+dj
                if ii<n and jj<n:
                    v=idx(ii,jj); Q[u,v]-=QL; Q[v,u]-=QL

    factors=[]
    for i in range(n):
        for j in range(n):
            inds=[idx(i,j)]; a=[1.0]
            for ii,jj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]:
                if 0<=ii<n and 0<=jj<n:
                    inds.append(idx(ii,jj)); a.append(-C_PDE)
            inds=np.asarray(inds,dtype=int); a=np.asarray(a)
            b=truth[i,j]
            for ii,jj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]:
                if 0<=ii<n and 0<=jj<n:
                    b -= C_PDE*truth[ii,jj]
            factors.append({'inds':inds,'a':a,'b':float(b),'site':(i,j)})
    return coords,truth,mean,Q,factors

coords, truth, mean_field, Q, factors = build_problem(N_GRID)
center=(N_GRID-1)/2
ACTION_I=range(int(center)-6,int(center)+7)
ACTION_J=range(int(center)-4,int(center)+5)
ACTION_IDS=np.array([i*N_GRID+j for i in ACTION_I for j in ACTION_J],dtype=int)

print(f'field dimension = {N_GRID**2}')
print(f'PDE factors N = {len(factors)}')
print(f'BO actions = {len(ACTION_IDS)}')

plt.figure(figsize=(6.5,5.0))
plt.imshow(truth,origin='lower')
plt.scatter([j for i in ACTION_I for j in ACTION_J],
            [i for i in ACTION_I for j in ACTION_J],
            s=6,alpha=.35,label='BO trust region')
plt.xlabel('grid j'); plt.ylabel('grid i')
plt.title('Reaction-diffusion field and localized BO search')
plt.legend(); plt.tight_layout(); plt.show()

## 3. Rigorous structural influence matrix

For \(r_j=a_j^\top f-b_j\),

\[
\left|\partial_i e_j\right|\le \frac{\gamma}{\tau}|a_{ji}|,
\qquad
\left|\partial_{ik}^2e_j\right|\le \frac{\gamma}{\tau^2}|a_{ji}a_{jk}|.
\]

Define

\[
A_{ii}=Q_{ii},
\]

\[
A_{ik}=-\left(|Q_{ik}|+\sum_j\frac{\gamma}{\tau^2}|a_{ji}a_{jk}|\right),\quad i\neq k.
\]

When \(A\succ0\), the coordinate-sensitive covariance inequality gives a valid influence operator \(A^{-1}\). EI is 1-Lipschitz, so the action-gap sensitivity has support only at the leader and challenger grid coordinates.

For omitted factors \(U\),

\[
B_{\rm struct}(x,\widehat x)=h_U^\top A^{-1}d(x,\widehat x),
\qquad
h_U=\sum_{j\in U}L(e_j).
\]

In [ ]:
def build_influence(Q, factors):
    d=Q.shape[0]
    kappa=np.abs(Q.copy()); np.fill_diagonal(kappa,0.0)
    L_factors=np.zeros((len(factors),d))
    hs=GAMMA/TAU**2
    for j,factor in enumerate(factors):
        inds=factor['inds']; a=factor['a']
        L_factors[j,inds]=(GAMMA/TAU)*np.abs(a)
        for p,u in enumerate(inds):
            for q,v in enumerate(inds):
                if u!=v:
                    kappa[u,v]+=hs*abs(a[p]*a[q])
    A=np.diag(np.diag(Q))-kappa
    return A,L_factors

A,L_FACTORS=build_influence(Q,factors)
lambda_min_A=np.linalg.eigvalsh(A)[0]
assert lambda_min_A>0
A_FACTOR=la.cho_factor(A,lower=True,check_finite=False)
print(f'lambda_min(A) = {lambda_min_A:.4f}')

## 4. Sparse non-Gaussian inference by GP-reference importance sampling

For the active set \(S\),

\[
\pi_S(df)\propto e^{-E_S(f)}P_0(df).
\]

Draw iid samples from \(P_0\) and weight them by \(e^{-E_S}\). This is proper non-Gaussian conditioning; the posterior is not replaced by a Gaussian approximation.

The Monte Carlo term below estimates the SNIS standard error of **acquisition gaps**, using common samples for the leader and challenger. It is a modular asymptotic inference allowance, not the structural theorem itself.

In [ ]:
def sample_reference(Q,mean_field,n_samples,seed):
    rng=np.random.default_rng(seed)
    L=np.linalg.cholesky(Q)
    z=rng.standard_normal((n_samples,Q.shape[0]))
    noise=la.solve_triangular(L.T,z.T,lower=False).T
    return (mean_field.ravel()+noise).astype(np.float32)

particles=sample_reference(Q,mean_field,N_PARTICLES,SEED)
EI_SAMPLES=np.maximum(particles[:,ACTION_IDS]-Y_BEST,0).astype(np.float32)
factor_cache={}

def factor_energy(j):
    if j not in factor_cache:
        factor=factors[j]
        residual=particles[:,factor['inds']]@factor['a']-factor['b']
        factor_cache[j]=(GAMMA*np.log(np.cosh(residual/TAU))).astype(np.float32)
    return factor_cache[j]

z_critical=st.norm.ppf(1-DELTA_MC/(2*len(ACTION_IDS)))

def sparse_inference(active):
    if active:
        E=np.zeros(N_PARTICLES,dtype=np.float32)
        for j in active: E+=factor_energy(j)
        logw=-E.astype(float)
    else:
        logw=np.zeros(N_PARTICLES)
    logw-=logw.max(); w=np.exp(logw); wn=w/w.sum()
    ess=w.sum()**2/(w@w)/N_PARTICLES
    acquisition=wn@EI_SAMPLES
    leader_local=int(np.argmax(acquisition)); leader_id=int(ACTION_IDS[leader_local])
    F=EI_SAMPLES-EI_SAMPLES[:,leader_local,None]
    gap=wn@F
    phi=(w[:,None]*(F-gap[None,:]))/w.mean()
    B_mc=z_critical*phi.std(axis=0,ddof=1)/np.sqrt(N_PARTICLES)
    return {'acquisition':acquisition,'gap':gap,'B_mc':B_mc,'ess':float(ess),
            'leader_id':leader_id,'leader_local':leader_local}

## 5. Optimize → challenge → refine → certify

At each iteration:

1. maximize the sparse EI;
2. construct the optimistic challenger envelope
   \[
   C_S(x)=\widehat\alpha_S(x)-\widehat\alpha_S(\widehat x)+B_{\rm struct}(x,\widehat x)+B_{\rm MC}(x,\widehat x);
   \]
3. find the worst challenger \(x_c\);
4. if \(\max_x C_S(x)\le\epsilon\), stop;
5. otherwise activate the omitted PDE factors with the largest contribution to the \((\widehat x,x_c)\) comparison.

In [ ]:
active=set(); history=[]; activation_batches=[]
for iteration in range(40):
    inf=sparse_inference(active)
    leader_id=inf['leader_id']; leader_local=inf['leader_local']
    omitted=[j for j in range(len(factors)) if j not in active]
    h_omitted=L_FACTORS[omitted].sum(axis=0) if omitted else np.zeros(N_GRID**2)
    w_struct=la.cho_solve(A_FACTOR,h_omitted,check_finite=False)
    B_struct=w_struct[ACTION_IDS]+w_struct[leader_id]
    B_struct[leader_local]=0.0
    envelope=inf['gap']+B_struct+inf['B_mc']; envelope[leader_local]=0.0
    challenger_local=int(np.argmax(envelope)); challenger_id=int(ACTION_IDS[challenger_local])
    epsilon=float(envelope[challenger_local])
    li,lj=np.unravel_index(leader_id,(N_GRID,N_GRID))
    ci,cj=np.unravel_index(challenger_id,(N_GRID,N_GRID))
    history.append({'iteration':iteration,'active_M':len(active),'leader_i':li,'leader_j':lj,
                    'challenger_i':ci,'challenger_j':cj,'epsilon_total':epsilon,
                    'challenger_gap':float(inf['gap'][challenger_local]),
                    'B_struct':float(B_struct[challenger_local]),
                    'B_MC':float(inf['B_mc'][challenger_local]),
                    'sparse_IS_ESS_fraction':inf['ess']})
    if epsilon<=EPS_TARGET: break
    d_decision=np.zeros(N_GRID**2); d_decision[leader_id]=1; d_decision[challenger_id]+=1
    v=la.cho_solve(A_FACTOR,d_decision,check_finite=False)
    contributions=L_FACTORS@v
    ranked=sorted(omitted,key=lambda j:contributions[j],reverse=True)
    to_add=ranked[:BATCH_ADD]
    activation_batches.append(to_add); active.update(to_add)

history_df=pd.DataFrame(history)
final=history_df.iloc[-1]
print(history_df)
print(f'Active factors: {len(active)} / {len(factors)}')
print(f'epsilon={final.epsilon_total:.4f}; structural={final.B_struct:.4f}; MC={final.B_MC:.4f}')
print(f'sparse GP-IS ESS={final.sparse_IS_ESS_fraction:.1%}')

In [ ]:
plt.figure(figsize=(7.2,4.7))
plt.plot(history_df.active_M,history_df.epsilon_total,marker='o',label='total certificate')
plt.plot(history_df.active_M,history_df.B_struct,marker='s',label='structural term')
plt.plot(history_df.active_M,history_df.B_MC,marker='^',label='inference term')
plt.axhline(EPS_TARGET,linestyle='--',label=f'target epsilon={EPS_TARGET}')
plt.xlabel('Active PDE factors M'); plt.ylabel('EI units')
plt.title('Adaptive decision certificate'); plt.legend(); plt.tight_layout(); plt.show()

activation=np.full(len(factors),np.nan)
for round_id,batch in enumerate(activation_batches,1):
    for j in batch: activation[j]=round_id
plt.figure(figsize=(6.5,5.0))
plt.imshow(np.ma.masked_invalid(activation.reshape(N_GRID,N_GRID)),origin='lower')
plt.scatter([int(final.leader_j)],[int(final.leader_i)],marker='*',s=140,label='leader')
plt.scatter([int(final.challenger_j)],[int(final.challenger_i)],marker='x',s=90,label='challenger')
plt.xlabel('grid j'); plt.ylabel('grid i'); plt.title('Adaptive PDE residual activation')
plt.legend(); plt.tight_layout(); plt.show()

## 6. Held-out full-posterior validation and sampler comparison

The adaptive algorithm has not evaluated the omitted PDE factors. For validation only, build the full posterior.

First measure how badly the original GP-reference proposal overlaps the full target. Then use a standard Laplace-preconditioned importance sampler:

1. compute the full-target MAP and Hessian \(H\);
2. sample from \(q(f)=\mathcal N(\widehat f,1.1H^{-1})\);
3. correct exactly with target/proposal importance weights.

This is not a new sampling claim; it is a clean independent validation of the DEC result.

In [ ]:
def all_factor_energy(samples, factor_list=factors):
    E=np.zeros(samples.shape[0])
    for factor in factor_list:
        residual=samples[:,factor['inds']]@factor['a']-factor['b']
        E+=GAMMA*np.log(np.cosh(residual/TAU))
    return E

E_full_reference=all_factor_energy(particles)
logw=-E_full_reference; logw-=logw.max(); w_ref_full=np.exp(logw)
full_GP_IS_ESS=w_ref_full.sum()**2/(w_ref_full@w_ref_full)/N_PARTICLES
print(f'Full-target GP-reference IS ESS = {full_GP_IS_ESS:.1%}')

def target_value_grad_hess(y,need_hess=True):
    dy=y-mean_field.ravel(); value=0.5*dy@(Q@dy); grad=Q@dy
    H=Q.copy() if need_hess else None
    for factor in factors:
        inds=factor['inds']; a=factor['a']; r=a@y[inds]-factor['b']; t=np.tanh(r/TAU)
        value+=GAMMA*np.log(np.cosh(r/TAU)); grad[inds]+=(GAMMA/TAU)*t*a
        if need_hess:
            H[np.ix_(inds,inds)]+=(GAMMA/TAU**2)*(1-t*t)*np.outer(a,a)
    return value,grad,H

y_map=mean_field.ravel().copy()
for iteration in range(10):
    value,grad,H=target_value_grad_hess(y_map,True)
    if np.linalg.norm(grad,np.inf)<1e-8: break
    step=np.linalg.solve(H,grad); directional=grad@step; step_size=1.0
    for _ in range(12):
        candidate=y_map-step_size*step
        candidate_value,_,_=target_value_grad_hess(candidate,False)
        if candidate_value<=value-1e-4*step_size*directional: break
        step_size*=0.5
    y_map=candidate
_,_,H=target_value_grad_hess(y_map,True)

INFLATION=1.10; rng=np.random.default_rng(801); L=np.linalg.cholesky(H/INFLATION)
z=rng.standard_normal((FULL_VALIDATION_PARTICLES,N_GRID**2))
full_samples=(y_map+la.solve_triangular(L.T,z.T,lower=False).T).astype(np.float32)
del z; gc.collect()

logw=np.empty(FULL_VALIDATION_PARTICLES)
for start in range(0,FULL_VALIDATION_PARTICLES,400):
    stop=min(FULL_VALIDATION_PARTICLES,start+400); sb=full_samples[start:stop].astype(float)
    dm=sb-y_map
    log_q=-0.5*np.einsum('bi,ij,bj->b',dm,H/INFLATION,dm,optimize=True)
    dp=sb-mean_field.ravel()
    log_target=-0.5*np.einsum('bi,ij,bj->b',dp,Q,dp,optimize=True)-all_factor_energy(sb)
    logw[start:stop]=log_target-log_q
logw-=logw.max(); w_full=np.exp(logw)
full_Laplace_IS_ESS=w_full.sum()**2/(w_full@w_full)/FULL_VALIDATION_PARTICLES
W_full=w_full/w_full.sum()
full_EI=np.maximum(full_samples[:,ACTION_IDS]-Y_BEST,0)
full_acquisition=W_full@full_EI
full_local=int(np.argmax(full_acquisition)); full_id=int(ACTION_IDS[full_local])
final_sparse=sparse_inference(active); DEC_id=final_sparse['leader_id']
DEC_local=int(np.where(ACTION_IDS==DEC_id)[0][0])
observed_regret=float(full_acquisition[full_local]-full_acquisition[DEC_local])

print(f'Full-target Laplace-IS ESS = {full_Laplace_IS_ESS:.1%}')
print('DEC action =',np.unravel_index(DEC_id,(N_GRID,N_GRID)))
print('full-posterior action =',np.unravel_index(full_id,(N_GRID,N_GRID)))
print(f'observed full EI regret = {observed_regret:.4f}')
print(f'certificate = {final.epsilon_total:.4f}')

## 7. Expanding-domain scaling experiment

The strongest intended regime is that the BO decision stays local while scientific information grows elsewhere. We enlarge the global PDE lattice while keeping the same central BO problem.

The key diagnostics are:

- total PDE factors \(N\);
- active factors \(M\) needed for a similar decision tolerance;
- GP-reference IS ESS for the sparse target;
- GP-reference IS ESS for the full target (post-hoc diagnostic only).

In [ ]:
def run_scaling_case(n,n_particles,seed,eps_target=0.075):
    _,_,mean_s,Q_s,factors_s=build_problem(n,right_bias=0.04)
    A_s,L_s=build_influence(Q_s,factors_s)
    A_s_factor=la.cho_factor(A_s,lower=True,check_finite=False)
    samples=sample_reference(Q_s,mean_s,n_particles,seed)
    center_s=(n-1)/2
    ii_s=range(max(0,int(center_s)-6),min(n,int(center_s)+7))
    jj_s=range(max(0,int(center_s)-4),min(n,int(center_s)+5))
    action_ids=np.array([i*n+j for i in ii_s for j in jj_s],dtype=int)
    utility=np.maximum(samples[:,action_ids]-Y_BEST,0).astype(np.float32)
    zc=st.norm.ppf(1-DELTA_MC/(2*len(action_ids)))
    cache={}
    def energy_j(j):
        if j not in cache:
            factor=factors_s[j]
            residual=samples[:,factor['inds']]@factor['a']-factor['b']
            cache[j]=(GAMMA*np.log(np.cosh(residual/TAU))).astype(np.float32)
        return cache[j]
    active_s=set()
    for _ in range(40):
        if active_s:
            E=np.zeros(n_particles,dtype=np.float32)
            for j in active_s: E+=energy_j(j)
            logw=-E.astype(float)
        else: logw=np.zeros(n_particles)
        logw-=logw.max(); w=np.exp(logw); wn=w/w.sum()
        sparse_ess=w.sum()**2/(w@w)/n_particles
        acquisition=wn@utility; leader_local=int(np.argmax(acquisition)); leader_id=int(action_ids[leader_local])
        F=utility-utility[:,leader_local,None]; gap=wn@F
        phi=(w[:,None]*(F-gap[None,:]))/w.mean(); B_mc=zc*phi.std(axis=0,ddof=1)/np.sqrt(n_particles)
        omitted=[j for j in range(len(factors_s)) if j not in active_s]
        h_omitted=L_s[omitted].sum(axis=0) if omitted else np.zeros(n*n)
        w_struct=la.cho_solve(A_s_factor,h_omitted,check_finite=False)
        B_struct=w_struct[action_ids]+w_struct[leader_id]; B_struct[leader_local]=0
        envelope=gap+B_struct+B_mc; envelope[leader_local]=0
        challenger_local=int(np.argmax(envelope)); challenger_id=int(action_ids[challenger_local])
        epsilon=float(envelope[challenger_local])
        if epsilon<=eps_target: break
        d=np.zeros(n*n); d[leader_id]=1; d[challenger_id]+=1
        v=la.cho_solve(A_s_factor,d,check_finite=False); contributions=L_s@v
        ranked=sorted(omitted,key=lambda j:contributions[j],reverse=True)
        active_s.update(ranked[:BATCH_ADD])
    E_full=np.zeros(n_particles)
    for factor in factors_s:
        residual=samples[:,factor['inds']]@factor['a']-factor['b']
        E_full+=GAMMA*np.log(np.cosh(residual/TAU))
    lw=-E_full; lw-=lw.max(); wf=np.exp(lw)
    full_ess=wf.sum()**2/(wf@wf)/n_particles
    return {'grid_n':n,'total_factors_N':len(factors_s),'active_factors_M':len(active_s),
            'epsilon':epsilon,'sparse_IS_ESS_fraction':sparse_ess,'full_IS_ESS_fraction':full_ess}

scaling=[]
for n_case,p_case in [(18,3000),(24,3000),(30,2500),(36,2200),(40,2000)]:
    scaling.append(run_scaling_case(n_case,p_case,1200+n_case))
scaling_df=pd.DataFrame(scaling)
print(scaling_df)

plt.figure(figsize=(7.2,4.7))
plt.plot(scaling_df.total_factors_N,scaling_df.active_factors_M,marker='o',label='DEC active factors')
plt.plot(scaling_df.total_factors_N,scaling_df.total_factors_N,linestyle='--',label='full conditioning')
plt.xlabel('Total PDE factors N'); plt.ylabel('Factors evaluated')
plt.title('Decision conditioning stays local'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(7.2,4.7))
plt.plot(scaling_df.total_factors_N,scaling_df.sparse_IS_ESS_fraction,marker='o',label='DEC sparse target')
plt.plot(scaling_df.total_factors_N,scaling_df.full_IS_ESS_fraction,marker='s',label='full PDE target')
plt.xlabel('Total PDE factors N'); plt.ylabel('GP-reference IS ESS fraction')
plt.title('Screening changes the inference regime'); plt.legend(); plt.tight_layout(); plt.show()

## 8. What this experiment establishes

A successful run supports three claims:

1. **Decision complexity can be much smaller than posterior complexity.** The global PDE factor count grows, while the number needed to settle the localized BO decision remains roughly fixed.
2. **The structural certificate is useful rather than merely finite.** The held-out full posterior agrees with or gives much smaller regret than the reported \(\epsilon\).
3. **Screening can change the inference regime.** The full target becomes increasingly difficult for GP-reference importance sampling, while the sparse decision-relevant target remains easy.

This suggests that the computational benefit is not only

\[
N\rightarrow M,
\]

but potentially

\[
\text{hard full non-Gaussian inference}
\rightarrow
\text{easy decision-sufficient non-Gaussian inference}.
\]

### Next stress test

The clean next step is to make the **PDE residual itself nonlinear** (for example a bounded semilinear reaction term) and compare HMC, tempered SMC, and FlowGP under the same action-level error budget.

In [ ]:
# Optional save block
history_df.to_csv(OUTPUT_DIR/'adaptive_history.csv',index=False)
scaling_df.to_csv(OUTPUT_DIR/'scaling_results.csv',index=False)
print('Saved outputs to',OUTPUT_DIR)